<h1>Chapter 7 - Retrieval</h1>
<i>Techniques to retrieve relevant information for your RAG system.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch07_retrieval/retrieval_techniques.ipynb)

---

This notebook is for Chapter 7 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


## Prerequisits

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to uncomment and run the following codeblock to install the dependencies for this chapter.

In [1]:
!pip install psycopg2==2.9.10
!pip install requests==2.32.3
!pip install openai==1.82.1

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for psycopg2: filename=psycopg2-2.9.10-cp312-cp312-linux_x86_64.whl size=529610 sha256=42e6d3b1cf32b1f6ae9dba68eeebf002109c8fb098c4dfb976b15c21acf27b72
  Stored in directory: /home/codespace/.cache/pip/wheels/ac/bb/ce/afa589c50b6004d3a06fc691e71bd09c9bd5f01e5921e5329b
Successfully built psycopg2
  Attempting uninstall: requests
    Found existing installation: requests 2.32.5
    Uninstalling requests-2.32.5:
      Successfully uninstalled requests-2.32.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 720.5/720.5 kB 7.8 MB/s  0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.83.0
    Uninstalling openai-1.83.0:
      Successfully uninstalled openai-1.83.0


### Load sample files

This notebook uses sample Word and PDF files.

When running the notebook on Google Colab, uncomment the code below to download the `datasets` directory from the Github repo.

In [ ]:
if IN_COLAB:
    !git clone --no-checkout https://github.com/polzerdo55862/RAG-with-Python-Cookbook.git
    %cd RAG-with-Python-Cookbook
    !git sparse-checkout init --cone
    !git sparse-checkout set datasets
    !git checkout
    !cp -r datasets /content/datasets

Cloning into 'RAG-with-Python-Cookbook'...


remote: Enumerating objects: 1334, done.
remote: Counting objects: 100% (319/319), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 1334 (delta 237), reused 198 (delta 173), pack-reused 1015 (from 1)
Receiving objects: 100% (1334/1334), 41.88 MiB | 11.19 MiB/s, done.
Resolving deltas: 100% (773/773), done.
/workspaces/RAG-with-Python-Cookbook/ch07_retrieval/RAG-with-Python-Cookbook
Your branch is up to date with 'origin/main'.
cp: cannot create directory '/content/datasets': No such file or directory


### Load secrets

If you run this code in Google Colab, save your OpenAI API key in the colab secrets and load it to the environmental variables.

In [3]:
from google.colab import userdata
import os

api_key = userdata.get("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found in Colab Secrets")

os.environ["OPENAI_API_KEY"] = api_key

ModuleNotFoundError: No module named 'google.colab'

### 1 Optimizing Query Results using Metadata Filtering in PostgreSQL

In [ ]:
# creates the vector extension if it does not exist
cur.execute("""CREATE EXTENSION IF NOT EXISTS vector""")

# Create table with metadata column
cur.execute(
    """
    CREATE TABLE IF NOT EXISTS embeddings_table_with_metadata (
        id SERIAL PRIMARY KEY,
        text_chunk TEXT,
        embedding VECTOR(1536),
        metadata JSONB
    )
"""
)
conn.commit()

In [ ]:
results

NameError: name 'results' is not defined

### 5.2 Enhancing Search Results by Extending the Original Query with Generated Pseudo-Documents

In [ ]:
# Define text chunks and metadata
text_chunks = [
    {
        "text": "Roger Federer has won 20 Grand Slam titles in tennis.",
        "topic": "tennis",
    },
    {
        "text": "The FIFA World Cup is the most prestigious "
        "football tournament.",
        "topic": "football",
    },
    {
        "text": "Serena Williams is one of the greatest tennis "
        "players of all time.",
        "topic": "tennis",
    },
    {
        "text": "Lionel Messi has won multiple Ballon d'Or "
        "awards in football.",
        "topic": "football",
    },
]
# Initialize OpenAI client
client = OpenAI()

# Insert text chunks with embeddings and metadata
for chunk in text_chunks:
    # Get embedding using OpenAI API
    response = client.embeddings.create(
        input=chunk["text"],
        model="text-embedding-3-small"
    )
    embedding = response.data[0].embedding

    metadata = {"topic": chunk["topic"]}
    cur.execute(
        "INSERT INTO embeddings_table_with_metadata "
        "(text_chunk, embedding, metadata) VALUES (%s, %s, %s)",
        (chunk["text"], embedding, json.dumps(metadata)),
    )

conn.commit()

In [ ]:
hypothetical_documents

["According to Company X's financial projections for 2024, the expected revenue is estimated to be $2 billion, reflecting a 15% growth compared to the previous year.",
 'In the Q1 2024 Earnings Report for Company X, the company reported a revenue of $500 million, and based on this quarterly trend, it is projected to reach $2.1 billion by the end of the fiscal year.',
 'Analyst forecasts for Company X suggest that with the introduction of new products and expanding market share, the total revenue for the year 2024 could potentially exceed $2.2 billion.']

### 5.3 Improving Search Results with Multi-Query Retrieval

In [ ]:
from openai import OpenAI
from pydantic import BaseModel
import os

client = OpenAI()

question = "What are the benefits of renewable energy?"

query_prompt = f"""You are an AI language model assistant. Your task is
to create three alternative versions of the provided user query to
enhance the retrieval of relevant documents from a vector database.
By offering diverse variations of the query, your goal is to help
mitigate the limitations of distance-based similarity search. Provide
these alternative queries, each on a new line.

Original query: {question}
"""

# send the query prompt to OpenAI
class QueryVariations(BaseModel):
    queries: list[str]

completion = client.beta.chat.completions.parse(
    model="gpt-5-mini",
    messages=[
        {
            "role": "user",
            "content": query_prompt,
        },
    ],
    response_format=QueryVariations,
)

queries = completion.choices[0].message.parsed.queries

In [ ]:
queries

['How does renewable energy positively impact the environment?',
 'What advantages does renewable energy offer compared to fossil fuels?',
 'Why is it beneficial to invest in renewable energy sources?']

### 5.4 Addressing Complex Requests by Designing a Query Routing System

In [ ]:
# Query the table with metadata filtering
query = "Who is the best player?"
response = client.embeddings.create(
    input=query,
    model="text-embedding-3-small"
)
query_embedding = response.data[0].embedding
topic_filter = "football"

cur.execute(
    f"""
    SELECT text_chunk, 1 - (embedding <=> %s::vector) AS similarity
    FROM embeddings_table_with_metadata
    WHERE metadata->>'topic' = %s
    ORDER BY similarity DESC
    LIMIT 5
    """,
    (query_embedding, topic_filter),
)
results = cur.fetchall()

In [ ]:
user_queries

[{'query': 'Who is the all-time top scorer in the FIFA World Cup?',
  'selected_data_source': 'general_football_knowledge'},
 {'query': 'What are the four Grand Slam tennis tournaments?',
  'selected_data_source': 'general_tennis_knowledge'},
 {'query': 'Did Manchester United win their last game?',
  'selected_data_source': 'latest_football_results_sql'}]

### 5.5 Increasing Search Efficiency by Designing an Auto-Merging Retriever (aka Parent Document Retriever)

In [ ]:
from pydantic import BaseModel
from openai import OpenAI

user_query = "What is the revenue of Company X in 2024?"

client = OpenAI()

class HypotheticalDocuments(BaseModel):
    documents: list[str]

prompt = f"""
You are an AI assistant. Based on the user query below, generate
three hypothetical text chunks that contain relevant information to
answer the query.
"""

completion = client.beta.chat.completions.parse(
    messages=[
        {"role": "system", "content": prompt},
        {"role": "user", "content": user_query},
    ],
    model="gpt-5-mini",
    response_format=HypotheticalDocuments,
)

hypothetical_documents = completion.choices[0].message.parsed.documents

### 5.6 Increasing Search Results by Designing a Sentence Window Retriever

In [ ]:
cur.execute(
    """
    CREATE TABLE sentence_window_retriever_text_chunks (
        chunk_id SERIAL PRIMARY KEY,
        chunk TEXT,
        chunk_embedding VECTOR(1536)
    )
"""
)
conn.commit()

### 5.7 Improving Search Accuracy with Reranking Methods

In [ ]:
from pydantic import BaseModel
from openai import OpenAI

client = OpenAI()

user_queries = [
    {
        "query": "Who is the all-time top scorer in the FIFA World Cup?",
        "selected_data_source": None,
    },
    {
        "query": "What are the four Grand Slam tennis tournaments?",
        "selected_data_source": None,
    },
    {
        "query": "Did Manchester United win their last game?",
        "selected_data_source": None,
    },
]

prompt = f"""
You are an expert at routing a user question to the appropriate
data source. Given a user question choose which of the data sources
in list_of_data_sources is the best to answer the question.
"""

from typing import Literal
from pydantic import Field

class RouterDecision(BaseModel):
    data_source: Literal[
        "general_football_knowledge",
        "general_tennis_knowledge",
        "latest_football_results_sql",
    ] = Field(
        ...,
        description="The best data source to answer the question."
    )

for user_query in user_queries:
    completion = client.beta.chat.completions.parse(
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": user_query["query"]},
        ],
        model="gpt-5-mini",
        response_format=RouterDecision,
    )
    user_query["selected_data_source"] = completion.choices[
        0
    ].message.parsed.data_source

In [ ]:
ranked_docs

RankedDocuments(text_chunk_ids=[1, 2, 5, 3, 4], relevance_scores=[5, 4, 3, 2, 1])

### 5.8 Decomposing Complex Queries into Multiple Sub-Queries

In [ ]:
file_path = (
    "../datasets/text_files/"
    "random-text-about-5-different-stories-with-paragraphs.txt"
)

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

leaf_node_size = 250  # size of the leaf nodes in characters
parent_merge = 4  # number of leaf nodes to merge into a parent node
parent_node_size = (
    parent_merge * leaf_node_size
)  # size of the parent nodes in characters
text_chunks = []  # list to store the text chunk dictionaries

for leaf_node_id in range(0, len(text) // leaf_node_size, 1):
    parent_node_id = leaf_node_id // parent_merge
    leaf_chunk_start = leaf_node_id * leaf_node_size
    parent_chunk_start = parent_node_id * parent_node_size

    text_chunk = {
        "leaf_node_id": leaf_node_id,
        "leaf_chunk": text[
            leaf_chunk_start : leaf_chunk_start + leaf_node_size
        ],
        "parent_node_id": parent_node_id,
        "parent_chunk": text[
            parent_chunk_start : parent_chunk_start + parent_node_size
        ],
    }
    text_chunks.append(text_chunk)

In [ ]:
decomposed_questions

[Question(question='What are the environmental benefits of renewable energy?', answer=None),
 Question(question='How does renewable energy impact public health compared to fossil fuels?', answer=None),
 Question(question='What are the economic benefits of using renewable energy?', answer=None),
 Question(question='How does the availability of renewable energy resources compare to fossil fuels?', answer=None)]